# **🐍 Intro to Python: Ch 2**

---
## **Financial Ratios from CSV 📈**

### **Concepts**
- Writing & reading CSV files with the `csv` module
- `pathlib.Path` and `with open()` for file handling
- Computing **Price-to-Book** and **P/E** ratios

| Ratio | Formula |
|-------|---------|
| Price-to-Book | Price ÷ Book Value per Share |
| P/E Ratio | Price ÷ Earnings per Share (EPS) |


### **2.1 Set up the data file**

Colab gives you a real filesystem. `Path("input.csv")` lands in `/content`, the working directory of your runtime.

When your data lives on your own machine instead, `files.upload()` is the right tool

In [4]:
#writes 'input.csv' to disk
from pathlib import Path

CSV_TEXT = """ticker,price,book_value_per_share,eps
RIL,2850.50,1425.25,45.67
TTMT,125.75,41.92,8.92
TSLA,245.30,20.44,3.64
WMT,82.40,13.73,5.73
MSFT,380.25,34.57,10.59
AAPL,178.50,3.97,6.05
GOOGL,140.80,23.47,5.23
AMZN,165.40,20.68,0.82
META,325.75,50.12,4.39
NFLX,445.60,49.51,3.48
"""

# Path() builds a filesystem path. write_text() creates the file and fills it,

data_path = Path("input.csv")
data_path.write_text(CSV_TEXT, encoding="utf-8")

print(f"✅ wrote {data_path.resolve()}")
print(f"   {data_path.stat().st_size} bytes, exists={data_path.exists()}")

✅ wrote C:\Users\najma\OneDrive\Desktop\Codebasics\Session_1_Resources\Session_1\input.csv
   281 bytes, exists=True


### **2.2 Read it back with `with open()`**

In [5]:
import csv

with open(data_path, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    rows = list(reader)

# Outside the block the file is already closed:
print(f"file closed: {f.closed}")
print(f"read {len(rows)} rows")
print(f"columns: {list(rows[0].keys())}")
print()
print("first row:", rows[0])

file closed: True
read 10 rows
columns: ['ticker', 'price', 'book_value_per_share', 'eps']

first row: {'ticker': 'RIL', 'price': '2850.50', 'book_value_per_share': '1425.25', 'eps': '45.67'}


### **2.3 Upload files to colab**

Upload `in.csv` when prompted. Note this cell waits for you to choose a file, so "Run all" will pause here until you do.

In [8]:
from google.colab import files

# This will prompt you to select the file from your local machine. Upload in.csv.
uploaded = files.upload()

Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "c:\Users\najma\OneDrive\Desktop\Codebasics\Session_1_Resources\Session_1\ch1_fastapi_intro-main\ch1_fastapi_intro-main\.venv\Lib\site-packages\IPython\core\interactiveshell.py", line 3823, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\najma\AppData\Local\Temp\ipykernel_18176\2899268802.py", line 1, in <module>
    from google.colab import files
ModuleNotFoundError: No module named 'google.colab'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\Users\najma\OneDrive\Desktop\Codebasics\Session_1_Resources\Session_1\ch1_fastapi_intro-main\ch1_fastapi_intro-main\.venv\Lib\site-packages\IPython\core\interactiveshell.py", line 2279, in showtraceback
    stb = self.InteractiveTB.structured_traceback(
        etype, value, tb, tb_offset=tb_offset
    )
  File "c:\Users\najma\OneDrive\Desktop\Codeb

In [ ]:
import io

filename = list(uploaded.keys())[0]
CSV_DATA = uploaded[filename].decode("utf-8")

print(f"✅ Loaded '{filename}'")
print(CSV_DATA[:300])

✅ Loaded 'in.csv'
ticker,price,book_value_per_share,eps
RIL,2850.5,1425.25,45.67
TTMT,125.75,41.92,8.92
TSLA,245.3,20.44,3.64
WMT,82.4,13.73,5.73
MSFT,380.25,34.57,10.59
AAPL,178.5,3.97,6.05
GOOGL,140.8,23.47,5.23
AMZN,165.4,20.68,0.82
META,325.75,50.12,4.39
NFLX,445.6,49.51,3.48



### **2.4 The `safe_ratio` helper**

Division has two ways to go wrong: the values might not be numbers, and the denominator might be zero. `safe_ratio` handles both and returns `""` when it can't produce an answer.

In [9]:
def safe_ratio(numerator: str, denominator: str) -> float | str:
    """Return numerator/denominator, or '' if invalid (zero division, bad type)."""
    try:
        num = float(numerator)
        den = float(denominator)
        if den == 0:
            return ""
        return round(num / den, 6)
    except (TypeError, ValueError):
        return ""

# Quick test
print(safe_ratio("2850.50", "1425.25"))   # RIL's P/B - expected : 2.0
print(safe_ratio("100", "0"))             # zero denominator - expected ''
print(safe_ratio("abc", "3"))             # not a numeric string - expected ''
print(safe_ratio("", "3"))                # empty cell in the CSV - expected ''


2.0





### **2.5 Compute the ratios for the Given CSV**

In [10]:
import csv, io

rows = []
reader = csv.DictReader(io.StringIO(CSV_DATA))

for row in reader:
    row["price_to_book_ratio"] = safe_ratio(row["price"], row["book_value_per_share"])
    row["pe_ratio"]            = safe_ratio(row["price"], row["eps"])
    rows.append(row)

# Pretty-print
header = f"{'Ticker':<8} {'P/B':>12} {'P/E':>12}"
print(header)
print("-" * len(header))
for r in rows:
    print(f"{r['ticker']:<8} {str(r['price_to_book_ratio']):>12} {str(r['pe_ratio']):>12}")

Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "c:\Users\najma\OneDrive\Desktop\Codebasics\Session_1_Resources\Session_1\ch1_fastapi_intro-main\ch1_fastapi_intro-main\.venv\Lib\site-packages\IPython\core\interactiveshell.py", line 3823, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\najma\AppData\Local\Temp\ipykernel_18176\3205377886.py", line 4, in <module>
    reader = csv.DictReader(io.StringIO(CSV_DATA))
                                        ^^^^^^^^
NameError: name 'CSV_DATA' is not defined

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\Users\najma\OneDrive\Desktop\Codebasics\Session_1_Resources\Session_1\ch1_fastapi_intro-main\ch1_fastapi_intro-main\.venv\Lib\site-packages\IPython\core\interactiveshell.py", line 2279, in showtraceback
    stb = self.InteractiveTB.structured_traceback(
        etype, value, tb, tb_offset=tb_off

### **2.6 Write the results out with `DictWriter`**

`DictReader` turned rows into dicts on the way in; `DictWriter` turns them back into rows on the way out.

One extra requirement: you must declare `fieldnames` up front, because a dict has no column order and a CSV does.


In [11]:
out_path = Path("out.csv")

fieldnames = ["ticker", "price", "book_value_per_share", "eps",
              "price_to_book_ratio", "pe_ratio"]

# "w" = write (creates the file, or replaces it entirely if it exists).
with open(out_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print(f"✅ wrote {out_path.resolve()} ({out_path.stat().st_size} bytes)")

✅ wrote C:\Users\najma\OneDrive\Desktop\Codebasics\Session_1_Resources\Session_1\out.csv (68 bytes)


In [ ]:
# Reading the file back in to verify it was written correctly
print(out_path.read_text(encoding="utf-8"))

with open(out_path, newline="", encoding="utf-8") as f:
    written = list(csv.DictReader(f))

assert len(written) == len(rows), f"expected {len(rows)} rows, found {len(written)}"
assert list(written[0].keys()) == fieldnames

ticker,price,book_value_per_share,eps,price_to_book_ratio,pe_ratio
RIL,2850.5,1425.25,45.67,2.0,62.415152
TTMT,125.75,41.92,8.92,2.999761,14.097534
TSLA,245.3,20.44,3.64,12.000978,67.39011
WMT,82.4,13.73,5.73,6.001457,14.380454
MSFT,380.25,34.57,10.59,10.999421,35.906516
AAPL,178.5,3.97,6.05,44.962217,29.504132
GOOGL,140.8,23.47,5.23,5.999148,26.921606
AMZN,165.4,20.68,0.82,7.998066,201.707317
META,325.75,50.12,4.39,6.499401,74.202733
NFLX,445.6,49.51,3.48,9.000202,128.045977



### **2.7 ✏️ Exercises**

1. **Highest P/E.** Which stock has it, and what does a high P/E imply?

2. **Earnings yield.** Add an `earnings_yield` column: `eps ÷ price`. This is P/E turned upside down — an earnings yield of 0.05 means 5% of the share price is earned per year. Which stock has the highest?

3. **A second output file.** Add your `earnings_yield` column to `fieldnames`, write the result to `out_extended.csv`, then read it back and `assert` the row count matches `rows`.

In [ ]:
# Your code here
